# 🔬 CL-SDRG: Full Pipeline
**Cross-Lingual Shortcut De-biasing via Reinforcement Learning Gating**

| Phase | Task | Est. Time |
|-------|------|-----------|
| 1 | Data Ingestion & Language ID | ~5-10 min |
| 2 | Architecture & VRAM Test | ~2 min |
| 3 | REINFORCE Training | ~20-30 min |
| 4 | Evaluation & Ablation | ~10-15 min |

**Runtime:** Set to **T4 GPU** → `Runtime → Change runtime type → T4`

---

## ⚙️ Setup & Dependencies

In [ ]:
!pip install -q torch transformers fasttext pandas numpy scikit-learn faiss-cpu rank_bm25 tqdm matplotlib seaborn
print('✅ Dependencies installed')

In [ ]:
import os, sys, random, time, logging, warnings
from pathlib import Path
from collections import Counter, defaultdict
from contextlib import contextmanager
from datetime import timedelta

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer

warnings.filterwarnings('ignore')

# ══════════════════════════════════════════════════════════════
#  CONFIGURATION
# ══════════════════════════════════════════════════════════════

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

# Paths
ROOT = Path('.')
DATA_DIR = ROOT / 'Fact Check Dataset'
CSV_PATH = DATA_DIR / 'claim_review.csv'
OUT = ROOT / 'outputs'
PROC = OUT / 'processed_data'
CKPT = OUT / 'checkpoints'
FIGS = OUT / 'figures'
LOGS = OUT / 'logs'
for d in [PROC, CKPT, FIGS, LOGS]: d.mkdir(parents=True, exist_ok=True)

# Model
ENCODER = 'intfloat/multilingual-e5-base'
EMB_DIM = 768
MAX_LEN = 128
FGA_H = 256
CLS_H = 256
CLS_DROP = 0.1
N_CLS = 3

# Training
LAM_ACC = 0.6; LAM_CONS = 0.4
LR = 1e-4; WD = 0.01
BS = 16; GRAD_ACCUM = 16
EPOCHS = 10; FP16 = True
BL_DECAY = 0.99; CKPT_EVERY = 2

# Eval
K_VALS = [1, 5, 20]; SFR_TARGET = 0.03; SFR_N = 10

# Labels
LABEL2ID = {'TRUE': 0, 'FALSE': 1, 'MIXED': 2}
ID2LABEL = {v:k for k,v in LABEL2ID.items()}

VERDICT_MAP = {
    'true':'TRUE','mostly true':'TRUE','correct':'TRUE','accurate':'TRUE',
    'verified':'TRUE','fact':'TRUE','confirmed':'TRUE',
    'false':'FALSE','mostly false':'FALSE','pants on fire':'FALSE',
    'pants on fire!':'FALSE','fake':'FALSE','incorrect':'FALSE',
    'not true':'FALSE','fabricated':'FALSE','debunked':'FALSE',
    'half true':'MIXED','half-true':'MIXED','mixture':'MIXED',
    'partially true':'MIXED','partially false':'MIXED',
    'misleading':'MIXED','unverified':'MIXED','unproven':'MIXED',
    'out of context':'MIXED','missing context':'MIXED',
    'exaggerated':'MIXED','needs context':'MIXED','distorts the facts':'MIXED',
    'falso':'FALSE','verdadeiro':'TRUE','enganoso':'MIXED',
    'impreciso':'MIXED','insustentável':'FALSE',
    'falsa':'FALSE','verdadero':'TRUE','engañoso':'MIXED','verdadera':'TRUE',
    'falsch':'FALSE','richtig':'TRUE','teilweise falsch':'MIXED',
    'irreführend':'MIXED','unbelegt':'MIXED',
    '\u091d\u0942\u0920':'FALSE','\u0938\u091a':'TRUE',
    '\u092d\u094d\u0930\u093e\u092e\u0915':'MIXED',
    '\u092b\u0930\u094d\u091c\u0940':'FALSE',
    '\u062c\u06be\u0648\u0679':'FALSE','\u0633\u0686':'TRUE',
    '\u06af\u0645\u0631\u0627\u06c1 \u06a9\u0646':'MIXED',
}

TARGET_LANGS = {'ur','hi','bn','pa','sd','ta','te','ml','mr','gu','ne','si'}
LID_CONF = 0.5; PAIR_DAYS = 3; MIN_URDU = 500
TRAIN_CUT = '2023-12-31'; TEST_START = '2024-01-01'

# Logging
logger = logging.getLogger()
logger.setLevel(logging.INFO)
logger.handlers.clear()
_h = logging.StreamHandler(sys.stdout)
_h.setFormatter(logging.Formatter('[%(asctime)s] %(levelname)-8s %(message)s', datefmt='%H:%M:%S'))
logger.addHandler(_h)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    logging.info(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_mem/1024**3:.1f} GB)')
else:
    logging.info('CPU mode')

def fmt(n): return f'{n:,}'

@contextmanager
def timer(label):
    t0 = time.perf_counter()
    logging.info(f'⏱  {label}')
    yield
    e = time.perf_counter() - t0
    logging.info(f'✅ {label} — {e:.1f}s' if e < 60 else f'✅ {label} — {int(e//60)}m {e%60:.1f}s')

def norm_verdict(raw):
    if not isinstance(raw, str) or not raw.strip(): return None
    c = raw.strip().lower()
    if c in VERDICT_MAP: return VERDICT_MAP[c]
    for k,v in VERDICT_MAP.items():
        if k in c or c in k: return v
    return None

print(f'✅ Config loaded | Device: {device} | vBatch: {BS*GRAD_ACCUM}')

## 📤 Upload Dataset

In [ ]:
if not CSV_PATH.exists():
    DATA_DIR.mkdir(exist_ok=True)
    from google.colab import files
    print('Upload claim_review.csv:')
    up = files.upload()
    for name, data in up.items():
        with open(str(CSV_PATH), 'wb') as f: f.write(data)
        print(f'Saved: {CSV_PATH} ({len(data)/1024**2:.1f} MB)')
else:
    print(f'Dataset found: {CSV_PATH} ({CSV_PATH.stat().st_size/1024**2:.1f} MB)')

---
# 📋 Phase 1: Data Ingestion & Language Identification
---

In [ ]:
# ── 1.1 Load & Sanitize ──
with timer('Loading CSV'):
    df = pd.read_csv(str(CSV_PATH), encoding='utf-8', low_memory=False, on_bad_lines='skip')
    logging.info(f'Raw: {fmt(len(df))} rows, {df.shape[1]} cols')

with timer('Zero-leakage sanitization'):
    for col in ['reviewRating.ratingExplanation','reviewRating.ratingValue']:
        if col in df.columns:
            logging.info(f'  Dropping: {col} ({fmt(df[col].notna().sum())} non-null)')
            df.drop(columns=[col], inplace=True)
    extra = [c for c in df.columns if 'explanation' in c.lower()]
    if extra: df.drop(columns=[c for c in extra if c in df.columns], inplace=True)

keep = ['claimReviewed','itemReviewed.author.name','datePublished','reviewRating.alternateName','author.name','url']
df = df[[c for c in keep if c in df.columns]].copy()
n0 = len(df)
df.dropna(subset=['claimReviewed','reviewRating.alternateName'], inplace=True)
df['claimReviewed'] = df['claimReviewed'].astype(str).str.strip()
df = df[df['claimReviewed'].str.len() > 5]
logging.info(f'  Sanitized: {fmt(len(df))} rows (dropped {fmt(n0-len(df))})')

In [ ]:
# ── 1.1b Verdict Normalization ──
with timer('Verdict normalization'):
    df['verdict'] = df['reviewRating.alternateName'].apply(norm_verdict)
    unmapped = df['verdict'].isna().sum()
    if unmapped > 0:
        logging.warning(f'  Unmapped: {fmt(unmapped)} rows')
        top_unmapped = df.loc[df['verdict'].isna(), 'reviewRating.alternateName'].value_counts().head(15)
        for lbl, cnt in top_unmapped.items(): logging.warning(f"    '{lbl}': {cnt}")
    df = df[df['verdict'].notna()].copy()
    df['label_id'] = df['verdict'].map(LABEL2ID)

print(f'\n  Label Distribution ({fmt(len(df))} total):')
for lbl, cnt in df['verdict'].value_counts().items():
    print(f'    {lbl}: {fmt(cnt)} ({cnt/len(df)*100:.1f}%)')

In [ ]:
# ── 1.2 FastText Language Identification ──
import fasttext
fasttext.FastText.eprint = lambda x: None

lid_path = OUT / 'lid.176.bin'
if not lid_path.exists():
    !wget -q https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin -O {str(lid_path)}
    print(f'Downloaded lid.176.bin')

with timer('FastText LID'):
    ft = fasttext.load_model(str(lid_path))
    langs, confs = [], []
    for text in tqdm(df['claimReviewed'].values, desc='LID', unit='claim'):
        clean = str(text).replace('\n',' ').replace('\r',' ').strip()
        if not clean:
            langs.append('unknown'); confs.append(0.0); continue
        p = ft.predict(clean, k=1)
        langs.append(p[0][0].replace('__label__',''))
        confs.append(float(p[1][0]))
    df['detected_lang'] = langs
    df['lang_confidence'] = confs
    del ft

print(f'\n  Top 20 Languages:')
for lang, cnt in df['detected_lang'].value_counts().head(20).items():
    print(f'    {lang}: {fmt(cnt)} ({cnt/len(df)*100:.1f}%)')

In [ ]:
# ── 1.2b South Asian filtering ──
df['is_south_asian'] = df['detected_lang'].isin(TARGET_LANGS)
sa_mask = df['is_south_asian'] & (df['lang_confidence'] >= LID_CONF)
sa_df = df[sa_mask].copy()
print(f'South Asian claims (conf≥{LID_CONF}):')
for lang, cnt in sa_df['detected_lang'].value_counts().items():
    print(f'  {lang}: {fmt(cnt)}')
print(f'Total: {fmt(len(sa_df))}')

# ── Silver pair mining ──
with timer(f'Silver pair mining (±{PAIR_DAYS}d)'):
    df['datePublished'] = pd.to_datetime(df['datePublished'], errors='coerce', utc=True)
    pdf = df.dropna(subset=['datePublished','author.name']).copy()
    pdf = pdf[pdf['author.name'].str.strip().str.len() > 0]
    pairs = []
    for org, grp in tqdm(pdf.groupby('author.name'), desc='Pairs', unit='org'):
        if len(grp)<2 or grp['detected_lang'].nunique()<2: continue
        grp = grp.sort_values('datePublished')
        dt,ln,cl = grp['datePublished'].values, grp['detected_lang'].values, grp['claimReviewed'].values
        for i in range(len(grp)):
            for j in range(i+1,len(grp)):
                d = abs((dt[j]-dt[i])/np.timedelta64(1,'D'))
                if d > PAIR_DAYS: break
                if ln[i]!=ln[j]: pairs.append({'claim_a':cl[i],'lang_a':ln[i],'claim_b':cl[j],'lang_b':ln[j],'org':org,'days':round(d,1)})
    silver = pd.DataFrame(pairs)
    print(f'  Silver pairs: {fmt(len(silver))}')

In [ ]:
# ── 1.3 Time-aware splits ──
with timer('Time-aware splitting'):
    valid = df['datePublished'].notna()
    df_d = df[valid].copy()
    train_df = df_d[df_d['datePublished'] <= pd.Timestamp(TRAIN_CUT, tz='UTC')].copy()
    test_df = df_d[df_d['datePublished'] >= pd.Timestamp(TEST_START, tz='UTC')].copy()

print(f'\n  Train (≤{TRAIN_CUT}): {fmt(len(train_df))}')
for v,c in train_df['verdict'].value_counts().items(): print(f'    {v}: {fmt(c)}')
print(f'  Test (≥{TEST_START}): {fmt(len(test_df))}')
for v,c in test_df['verdict'].value_counts().items(): print(f'    {v}: {fmt(c)}')

# Smoke test S1
urdu_n = (df['detected_lang']=='ur').sum()
print(f'\n  Urdu claims: {fmt(urdu_n)} {"✅ PASS" if urdu_n>=MIN_URDU else "⚠️ FAIL (need NLLB)"}')

# Save
with timer('Saving processed data'):
    df.to_csv(str(PROC/'fci_processed_full.csv'), index=False)
    train_df.to_csv(str(PROC/'fci_train.csv'), index=False)
    test_df.to_csv(str(PROC/'fci_test.csv'), index=False)
    sa_df.to_csv(str(PROC/'fci_south_asian.csv'), index=False)
    if len(silver)>0: silver.to_csv(str(PROC/'silver_pairs.csv'), index=False)

print(f'\n{"="*70}')
print(f'  ✅ PHASE 1 COMPLETE')
print(f'  Processed: {fmt(len(df))} | Train: {fmt(len(train_df))} | Test: {fmt(len(test_df))}')
print(f'  SA claims: {fmt(len(sa_df))} | Silver pairs: {fmt(len(silver))} | Urdu: {fmt(urdu_n)}')
print(f'{"="*70}')

---
# 🏗️ Phase 2: Architecture & VRAM Validation
---

In [ ]:
# ══════════════════════════════════════════════════════════════
#  MODEL ARCHITECTURE
# ══════════════════════════════════════════════════════════════

class FrozenEncoder(nn.Module):
    """Frozen multilingual-E5-base with mean pooling."""
    def __init__(self, name=ENCODER):
        super().__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(name)
        self.encoder = AutoModel.from_pretrained(name)
        for p in self.encoder.parameters(): p.requires_grad = False
        self.encoder.eval()
        logging.info(f'Encoder loaded: {name} | Frozen params: {fmt(sum(p.numel() for p in self.encoder.parameters()))}')

    @torch.no_grad()
    def encode(self, texts, dev):
        tok = self.tokenizer([f'query: {str(t)}' for t in texts], max_length=MAX_LEN,
                             padding=True, truncation=True, return_tensors='pt').to(dev)
        with torch.amp.autocast('cuda', enabled=dev.type=='cuda'):
            out = self.encoder(**tok)
        m = tok['attention_mask'].unsqueeze(-1).float()
        return (out.last_hidden_state * m).sum(1) / m.sum(1).clamp(min=1e-9)


class FeatureGatingAgent(nn.Module):
    """FGA: [E_q;E_s;E_t] → MLP → Sigmoid → (α_q, α_s, α_t)"""
    def __init__(self):
        super().__init__()
        self.dim = EMB_DIM
        self.net = nn.Sequential(
            nn.Linear(EMB_DIM*3, FGA_H), nn.ReLU(True),
            nn.Linear(FGA_H, EMB_DIM*3), nn.Sigmoid())
        logging.info(f'FGA: {fmt(sum(p.numel() for p in self.parameters()))} params')

    def forward(self, eq, es, et):
        return self.net(torch.cat([eq,es,et],-1)).split(self.dim, -1)


class GatedFusion(nn.Module):
    """E_gated = α_q⊙E_q + α_s⊙E_s + α_t⊙E_t"""
    def forward(self, eq,es,et,aq,a_s,at):
        return aq*eq + a_s*es + at*et


class VeracityClassifier(nn.Module):
    """LayerNorm → FC → ReLU → Dropout → FC → logits"""
    def __init__(self):
        super().__init__()
        self.clf = nn.Sequential(
            nn.LayerNorm(EMB_DIM), nn.Linear(EMB_DIM, CLS_H),
            nn.ReLU(True), nn.Dropout(CLS_DROP), nn.Linear(CLS_H, N_CLS))
        logging.info(f'Classifier: {fmt(sum(p.numel() for p in self.parameters()))} params')

    def forward(self, x): return self.clf(x)


print('✅ Architecture defined')

In [ ]:
# ── VRAM Dry-Run ──
if device.type == 'cuda':
    torch.cuda.reset_peak_memory_stats(); torch.cuda.empty_cache()

enc_test = FrozenEncoder().to(device)
fga_test = FeatureGatingAgent().to(device)
fus_test = GatedFusion().to(device)
clf_test = VeracityClassifier().to(device)

total_p = sum(p.numel() for m in [enc_test.encoder, fga_test, clf_test] for p in m.parameters())
train_p = sum(p.numel() for m in [fga_test, clf_test] for p in m.parameters() if p.requires_grad)
print(f'\n  Total: {fmt(total_p)} | Trainable: {fmt(train_p)} ({train_p/total_p*100:.2f}%)')

# Forward pass
dummy = [f'Test claim {i}' for i in range(BS)]
with torch.no_grad():
    if device.type=='cuda':
        with torch.cuda.amp.autocast():
            eq = enc_test.encode(dummy, device)
            es = enc_test.encode([f'Speaker {i}' for i in range(BS)], device)
            et = enc_test.encode([f'2023-01-{i+1:02d}' for i in range(BS)], device)
            aq,a_s,at = fga_test(eq,es,et)
            logits = clf_test(fus_test(eq,es,et,aq,a_s,at))
    else:
        eq = enc_test.encode(dummy, device)
        es = enc_test.encode([f'Speaker {i}' for i in range(BS)], device)
        et = enc_test.encode([f'2023-01-{i+1:02d}' for i in range(BS)], device)
        aq,a_s,at = fga_test(eq,es,et)
        logits = clf_test(fus_test(eq,es,et,aq,a_s,at))

print(f'  Logits: {logits.shape}')
if device.type == 'cuda':
    peak = torch.cuda.max_memory_allocated()/1024**3
    print(f'  Peak VRAM: {peak:.2f} GB {"✅ PASS" if peak<2.5 else "⚠️ FAIL"} (budget: 2.5 GB)')

del enc_test, fga_test, fus_test, clf_test, eq, es, et
torch.cuda.empty_cache() if device.type=='cuda' else None

print(f'\n{"="*70}\n  ✅ PHASE 2 COMPLETE — Architecture validated\n{"="*70}')

---
# 🎯 Phase 3: REINFORCE Training
---

In [ ]:
# ── Load training data ──
tr = pd.read_csv(str(PROC/'fci_train.csv'))
tr['itemReviewed.author.name'] = tr['itemReviewed.author.name'].fillna('Unknown')
tr = tr.dropna(subset=['claimReviewed','label_id'])
tr['label_id'] = tr['label_id'].astype(int)
logging.info(f'Train samples: {fmt(len(tr))}')
for lid, cnt in tr['label_id'].value_counts().sort_index().items():
    logging.info(f'  {ID2LABEL.get(lid,"?")}: {fmt(cnt)}')

In [ ]:
# ── Pre-compute embeddings ──
encoder = FrozenEncoder().to(device)

def enc_all(enc, texts, desc, bs=64):
    embs = []
    for i in tqdm(range(0,len(texts),bs), desc=desc, unit='b'):
        embs.append(enc.encode(texts[i:i+bs], device).cpu().float())
    return torch.cat(embs)

claims_list = tr['claimReviewed'].tolist()
speakers_list = tr['itemReviewed.author.name'].tolist()
dates_list = tr['datePublished'].astype(str).tolist()
labels_t = torch.tensor(tr['label_id'].values, dtype=torch.long)

t0 = time.time()
tr_ce = enc_all(encoder, claims_list, 'Train claims')

# Dedup speakers
uniq_sp = list(set(speakers_list))
logging.info(f'Unique speakers: {fmt(len(uniq_sp))}')
sp_emb_map = {}
for i in tqdm(range(0,len(uniq_sp),64), desc='Speakers'):
    b = uniq_sp[i:i+64]
    e = encoder.encode(b, device).cpu().float()
    for s, emb in zip(b, e): sp_emb_map[s] = emb
tr_se = torch.stack([sp_emb_map[s] for s in speakers_list])

tr_de = enc_all(encoder, dates_list, 'Train dates')
logging.info(f'Embeddings: {time.time()-t0:.0f}s | shapes: {tr_ce.shape} {tr_se.shape} {tr_de.shape}')

del encoder; torch.cuda.empty_cache() if device.type=='cuda' else None

In [ ]:
# ── Dataset + DataLoader ──
class EmbDS(Dataset):
    def __init__(self, ce,se,de,lbl,sp,smap):
        self.ce,self.se,self.de,self.lbl,self.sp = ce,se,de,lbl,sp
        self.smap = smap; self.all_sp = list(smap.keys())
    def __len__(self): return len(self.lbl)
    def __getitem__(self, i):
        return {'ce':self.ce[i],'se':self.se[i],'de':self.de[i],'lbl':self.lbl[i],'sp':self.sp[i]}
    def cf(self, s):
        c = [x for x in self.all_sp if x!=s]
        return self.smap[random.choice(c)] if c else self.smap[s]

ds = EmbDS(tr_ce, tr_se, tr_de, labels_t, speakers_list, sp_emb_map)

def collate(batch):
    return {'ce': torch.stack([b['ce'] for b in batch]),
            'se': torch.stack([b['se'] for b in batch]),
            'de': torch.stack([b['de'] for b in batch]),
            'lbl': torch.stack([b['lbl'] for b in batch]),
            'cf_se': torch.stack([ds.cf(b['sp']) for b in batch])}

dl = DataLoader(ds, batch_size=BS, shuffle=True, num_workers=0, collate_fn=collate, drop_last=True)
logging.info(f'DataLoader: {len(dl)} batches × {BS}')

In [ ]:
# ══════════════════════════════════════════════════════════════
#  REINFORCE TRAINING LOOP
# ══════════════════════════════════════════════════════════════

fga = FeatureGatingAgent().to(device)
fusion = GatedFusion().to(device)
classifier = VeracityClassifier().to(device)

params = list(fga.parameters()) + list(classifier.parameters())
optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=WD)
scaler = torch.amp.GradScaler('cuda', enabled=FP16 and device.type=='cuda')

baseline = 0.0
history = []

print(f'\n{"="*70}')
print(f'  REINFORCE Training | {EPOCHS} epochs | λ_acc={LAM_ACC} λ_cons={LAM_CONS}')
print(f'{"="*70}\n')

for epoch in range(EPOCHS):
    fga.train(); classifier.train()
    ep_loss, ep_rew, ep_ra, ep_rc = [], [], [], []
    correct = total = 0
    optimizer.zero_grad()

    pbar = tqdm(dl, desc=f'Epoch {epoch+1}/{EPOCHS}', unit='b')
    for step, batch in enumerate(pbar):
        eq = batch['ce'].to(device)
        es = batch['se'].to(device)
        et = batch['de'].to(device)
        es_cf = batch['cf_se'].to(device)
        tgt = batch['lbl'].to(device)

        with torch.amp.autocast('cuda', enabled=FP16 and device.type=='cuda'):
            aq,a_s,at = fga(eq,es,et)
            logits = classifier(fusion(eq,es,et,aq,a_s,at))
            aq2,as2,at2 = fga(eq,es_cf,et)
            logits_cf = classifier(fusion(eq,es_cf,et,aq2,as2,at2))

            probs = F.softmax(logits,-1); probs_cf = F.softmax(logits_cf,-1)
            preds = logits.argmax(-1)

            r_acc = (preds==tgt).float()*2-1
            r_cons = 1.0 - torch.abs(probs-probs_cf).sum(-1)
            r_total = LAM_ACC*r_acc + LAM_CONS*r_cons

            log_p = F.log_softmax(logits,-1).gather(1,preds.unsqueeze(1)).squeeze(1)
            policy = -torch.mean((r_total-baseline).detach() * log_p)
            ce = F.cross_entropy(logits, tgt)
            loss = (policy + 0.5*ce) / GRAD_ACCUM

        scaler.scale(loss).backward()
        if (step+1)%GRAD_ACCUM==0 or (step+1)==len(dl):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad()

        br = r_total.mean().item()
        baseline = BL_DECAY*baseline + (1-BL_DECAY)*br
        ep_loss.append(loss.item()*GRAD_ACCUM); ep_rew.append(br)
        ep_ra.append(r_acc.mean().item()); ep_rc.append(r_cons.mean().item())
        correct += (preds==tgt).sum().item(); total += len(tgt)
        pbar.set_postfix(loss=f'{np.mean(ep_loss[-50:]):.4f}', rew=f'{np.mean(ep_rew[-50:]):.3f}', acc=f'{correct/total:.3f}')

    m = {'epoch': epoch+1, 'loss': float(np.mean(ep_loss)), 'reward': float(np.mean(ep_rew)),
         'accuracy': correct/total, 'r_acc': float(np.mean(ep_ra)), 'r_cons': float(np.mean(ep_rc))}
    history.append(m)
    logging.info(f'  Loss={m["loss"]:.4f} Rew={m["reward"]:.3f} Acc={m["accuracy"]:.3f} R_acc={m["r_acc"]:.3f} R_cons={m["r_cons"]:.3f}')

    if (epoch+1)%CKPT_EVERY==0 or (epoch+1)==EPOCHS:
        torch.save({'epoch':epoch+1, 'model_state_dict':{'fga':fga.state_dict(),'classifier':classifier.state_dict()},
                     'optimizer_state_dict':optimizer.state_dict(), 'metrics':m},
                    str(CKPT/f'cl_sdrg_epoch_{epoch+1}.pt'))
        logging.info(f'  💾 Checkpoint: epoch {epoch+1}')

pd.DataFrame(history).to_csv(str(LOGS/'training_history.csv'), index=False)

In [ ]:
# ── Training Curves ──
import matplotlib.pyplot as plt

eps = range(1, len(history)+1)
fig, ax = plt.subplots(2,2, figsize=(14,10))
fig.suptitle('CL-SDRG Training Progress', fontsize=16, fontweight='bold')
ax[0,0].plot(eps,[h['loss'] for h in history],'b-o',lw=2); ax[0,0].set_title('Loss'); ax[0,0].grid(alpha=.3)
ax[0,1].plot(eps,[h['reward'] for h in history],'g-o',lw=2); ax[0,1].set_title('Total Reward'); ax[0,1].grid(alpha=.3)
ax[1,0].plot(eps,[h['r_acc'] for h in history],'r-s',label='R_acc',lw=2)
ax[1,0].plot(eps,[h['r_cons'] for h in history],'m-^',label='R_cons',lw=2)
ax[1,0].set_title('Component Rewards'); ax[1,0].legend(); ax[1,0].grid(alpha=.3)
ax[1,1].plot(eps,[h['accuracy'] for h in history],'c-D',lw=2); ax[1,1].set_title('Accuracy'); ax[1,1].grid(alpha=.3)
plt.tight_layout()
plt.savefig(str(FIGS/'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'\n{"="*70}')
print(f'  ✅ PHASE 3 COMPLETE | Best Acc: {max(h["accuracy"] for h in history):.4f} | Best Reward: {max(h["reward"] for h in history):.4f}')
print(f'{"="*70}')

---
# 📊 Phase 4: Evaluation, Ablation & SFR Audit
---

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix
import faiss

# ── Load test data & embeddings ──
te = pd.read_csv(str(PROC/'fci_test.csv'))
te['itemReviewed.author.name'] = te['itemReviewed.author.name'].fillna('Unknown')
te = te.dropna(subset=['claimReviewed','label_id'])
te['label_id'] = te['label_id'].astype(int)
logging.info(f'Test: {fmt(len(te))}')

enc_eval = FrozenEncoder().to(device)

te_claims = te['claimReviewed'].tolist()
te_speakers = te['itemReviewed.author.name'].tolist()
te_dates = te['datePublished'].astype(str).tolist()
te_labels = te['label_id'].values

te_ce = enc_all(enc_eval, te_claims, 'Test claims')
te_se = enc_all(enc_eval, te_speakers, 'Test speakers')
te_de = enc_all(enc_eval, te_dates, 'Test dates')

# Speaker map for SFR
all_sp_eval = list(set(te_speakers + speakers_list))
if len(all_sp_eval)>5000: all_sp_eval = random.sample(all_sp_eval, 5000)
sp_eval_map = {}
for i in range(0,len(all_sp_eval),64):
    b = all_sp_eval[i:i+64]
    e = enc_eval.encode(b, device).cpu().float()
    for s, emb in zip(b,e): sp_eval_map[s] = emb

# Train embeddings for baselines
tr_for_bl = pd.read_csv(str(PROC/'fci_train.csv'))
tr_for_bl = tr_for_bl.dropna(subset=['claimReviewed','label_id'])
tr_bl_claims = tr_for_bl['claimReviewed'].tolist()
tr_bl_labels = tr_for_bl['label_id'].astype(int).values
tr_bl_ce = enc_all(enc_eval, tr_bl_claims, 'Train claims (baseline)')

del enc_eval; torch.cuda.empty_cache() if device.type=='cuda' else None

In [ ]:
# ── 4.1 Classification ──
fga.eval(); classifier.eval()
all_preds, all_probs = [], []
with torch.no_grad():
    for i in range(0,len(te_ce),BS):
        eq=te_ce[i:i+BS].to(device); es=te_se[i:i+BS].to(device); et=te_de[i:i+BS].to(device)
        aq,a_s,at = fga(eq,es,et)
        logits = classifier(fusion(eq,es,et,aq,a_s,at))
        all_preds.append(logits.argmax(-1).cpu().numpy())
        all_probs.append(F.softmax(logits,-1).cpu().numpy())

y_pred = np.concatenate(all_preds)
y_prob = np.concatenate(all_probs)
names = [ID2LABEL[i] for i in range(N_CLS)]

print('\n📊 Classification Report:')
print(classification_report(te_labels, y_pred, target_names=names, zero_division=0))

cls_m = {
    'accuracy': accuracy_score(te_labels, y_pred),
    'macro_f1': f1_score(te_labels, y_pred, average='macro', zero_division=0),
    'macro_precision': precision_score(te_labels, y_pred, average='macro', zero_division=0),
    'macro_recall': recall_score(te_labels, y_pred, average='macro', zero_division=0),
}
cm = confusion_matrix(te_labels, y_pred)

In [ ]:
# ── 4.1b Retrieval (FAISS) ──
gated_e = []
with torch.no_grad():
    for i in range(0,len(te_ce),BS):
        eq=te_ce[i:i+BS].to(device); es=te_se[i:i+BS].to(device); et=te_de[i:i+BS].to(device)
        aq,a_s,at=fga(eq,es,et)
        gated_e.append(fusion(eq,es,et,aq,a_s,at).cpu().numpy())
gated = np.concatenate(gated_e).astype(np.float32)

gc = gated.copy(); faiss.normalize_L2(gc)
idx = faiss.IndexFlatIP(EMB_DIM); idx.add(gc)
qc = gated.copy(); faiss.normalize_L2(qc)
D, I = idx.search(qc, max(K_VALS)+1)

ret_m = {}
for k in K_VALS:
    rec,mrr,ndcg = [],[],[]
    for i in range(len(qc)):
        rl = te_labels[I[i]]; mask = I[i]!=i; rl = rl[mask][:k]
        rel = rl==te_labels[i]
        rec.append(float(rel.any()))
        cp = np.where(rel)[0]; mrr.append(1.0/(cp[0]+1) if len(cp)>0 else 0.0)
        dcg = sum(r/np.log2(p+2) for p,r in enumerate(rel.astype(float)))
        idcg = sum(1.0/np.log2(p+2) for p in range(max(1,int(rel.sum()))))
        ndcg.append(dcg/idcg if idcg>0 else 0.0)
    ret_m[f'recall@{k}']=float(np.mean(rec))
    ret_m[f'mrr@{k}']=float(np.mean(mrr))
    ret_m[f'ndcg@{k}']=float(np.mean(ndcg))

print('\n📊 Retrieval Metrics:')
for k in K_VALS:
    print(f'  K={k:<4} Recall={ret_m[f"recall@{k}"]:.4f}  MRR={ret_m[f"mrr@{k}"]:.4f}  nDCG={ret_m[f"ndcg@{k}"]:.4f}')

In [ ]:
# ── 4.2 Speaker Flip Rate (SFR) ──
print(f'\n🔍 SFR Audit: {fmt(len(te_ce))} × {SFR_N} perturbations')
sp_list = list(sp_eval_map.keys())
n_te = len(te_ce)
flips = np.zeros(n_te)

with torch.no_grad():
    orig_p = []
    for i in range(0,n_te,64):
        eq=te_ce[i:i+64].to(device); es=te_se[i:i+64].to(device); et=te_de[i:i+64].to(device)
        aq,a_s,at=fga(eq,es,et)
        orig_p.append(classifier(fusion(eq,es,et,aq,a_s,at)).argmax(-1).cpu())
    orig_p = torch.cat(orig_p)

    for pert in tqdm(range(SFR_N), desc='SFR'):
        rsp = random.choices(sp_list, k=n_te)
        cf_se = torch.stack([sp_eval_map[s] for s in rsp])
        cf_p = []
        for i in range(0,n_te,64):
            eq=te_ce[i:i+64].to(device); es_cf=cf_se[i:i+64].to(device); et=te_de[i:i+64].to(device)
            aq,a_s,at=fga(eq,es_cf,et)
            cf_p.append(classifier(fusion(eq,es_cf,et,aq,a_s,at)).argmax(-1).cpu())
        cf_p = torch.cat(cf_p)
        flips += (orig_p!=cf_p).numpy().astype(float)

sfr_val = float(np.mean(flips/SFR_N))
sfr_pass = sfr_val < SFR_TARGET
print(f'  SFR = {sfr_val:.4f} ({sfr_val*100:.2f}%) {"✅ PASS" if sfr_pass else "⚠️ FAIL"} (target <{SFR_TARGET*100:.0f}%)')

In [ ]:
# ── 4.3 Ablation Baselines ──
abl = []

# BM25
try:
    from rank_bm25 import BM25Okapi
    bm25 = BM25Okapi([c.lower().split() for c in tr_bl_claims])
    bp = np.array([tr_bl_labels[bm25.get_scores(c.lower().split()).argmax()] for c in tqdm(te_claims, desc='BM25')])
    abl.append({'method':'BM25','accuracy':accuracy_score(te_labels,bp),
                'macro_f1':f1_score(te_labels,bp,average='macro',zero_division=0),
                'macro_precision':precision_score(te_labels,bp,average='macro',zero_division=0),
                'macro_recall':recall_score(te_labels,bp,average='macro',zero_division=0)})
    print(f'BM25: Acc={abl[-1]["accuracy"]:.4f} F1={abl[-1]["macro_f1"]:.4f}')
except: print('BM25 skipped')

# Zero-Shot kNN
tr_np=tr_bl_ce.numpy().astype(np.float32); te_np=te_ce.numpy().astype(np.float32)
trc=tr_np.copy(); faiss.normalize_L2(trc)
tec=te_np.copy(); faiss.normalize_L2(tec)
zidx=faiss.IndexFlatIP(EMB_DIM); zidx.add(trc)
_,zI=zidx.search(tec,1)
zp=tr_bl_labels[zI[:,0]]
abl.append({'method':'Zero-Shot kNN','accuracy':accuracy_score(te_labels,zp),
            'macro_f1':f1_score(te_labels,zp,average='macro',zero_division=0),
            'macro_precision':precision_score(te_labels,zp,average='macro',zero_division=0),
            'macro_recall':recall_score(te_labels,zp,average='macro',zero_division=0)})
print(f'Zero-Shot kNN: Acc={abl[-1]["accuracy"]:.4f} F1={abl[-1]["macro_f1"]:.4f}')

In [ ]:
# ══════════════════════════════════════════════════════════════
#  FINAL RESULTS TABLE
# ══════════════════════════════════════════════════════════════

print(f'\n{"═"*90}')
print(f'  📊 BENCHMARK RESULTS — CL-SDRG')
print(f'{"═"*90}')
print(f'{"Method":<30} {"Acc":>8} {"F1":>8} {"Prec":>8} {"Rec":>8} {"SFR":>8}')
print(f'{"-"*90}')
print(f'{"CL-SDRG (Ours)":<30} {cls_m["accuracy"]:8.4f} {cls_m["macro_f1"]:8.4f} '
      f'{cls_m["macro_precision"]:8.4f} {cls_m["macro_recall"]:8.4f} {sfr_val:8.4f}')
for r in abl:
    print(f'{r["method"]:<30} {r["accuracy"]:8.4f} {r["macro_f1"]:8.4f} '
          f'{r["macro_precision"]:8.4f} {r["macro_recall"]:8.4f} {"N/A":>8}')
print(f'{"-"*90}')

print(f'\n  Retrieval (CL-SDRG):')
for k in K_VALS:
    print(f'    K={k:<4} Recall={ret_m[f"recall@{k}"]:.4f}  MRR={ret_m[f"mrr@{k}"]:.4f}  nDCG={ret_m[f"ndcg@{k}"]:.4f}')

print(f'\n  SFR: {sfr_val:.4f} ({sfr_val*100:.2f}%) — {"✅ TARGET MET" if sfr_pass else "⚠️ TARGET NOT MET"}')
print(f'{"═"*90}')
print(f'\n🎉 COPY EVERYTHING ABOVE AND SHARE BACK!')

In [ ]:
# ── Plots ──
fig, axes = plt.subplots(1,3, figsize=(20,6))
fig.suptitle('CL-SDRG Evaluation', fontsize=16, fontweight='bold')

# Classification bars
methods = ['CL-SDRG']+[r['method'] for r in abl]
all_r = [cls_m]+abl
mn = ['accuracy','macro_f1','macro_precision','macro_recall']
ml = ['Accuracy','F1','Precision','Recall']
x = np.arange(len(mn)); w = 0.8/len(methods)
colors = plt.cm.Set2(np.linspace(0,1,len(methods)))
for i,(meth,r) in enumerate(zip(methods,all_r)):
    axes[0].bar(x+i*w, [r.get(m,0) for m in mn], w, label=meth, color=colors[i])
axes[0].set_xticks(x+w*(len(methods)-1)/2); axes[0].set_xticklabels(ml)
axes[0].set_ylim(0,1); axes[0].legend(fontsize=8); axes[0].set_title('Classification'); axes[0].grid(axis='y',alpha=.3)

# SFR
b = axes[1].bar(['CL-SDRG'],[sfr_val*100],color=['#2ecc71'])
axes[1].axhline(SFR_TARGET*100,color='red',ls='--',lw=2,label=f'Target {SFR_TARGET*100:.0f}%')
axes[1].set_ylabel('SFR (%)'); axes[1].set_title('Speaker Flip Rate'); axes[1].legend()
axes[1].text(0,sfr_val*100+0.3,f'{sfr_val*100:.2f}%',ha='center',fontweight='bold')

# Confusion matrix
im = axes[2].imshow(cm,cmap='Blues')
axes[2].set_title('Confusion Matrix')
lbls = [ID2LABEL[i] for i in range(N_CLS)]
axes[2].set_xticks(range(len(lbls))); axes[2].set_yticks(range(len(lbls)))
axes[2].set_xticklabels(lbls); axes[2].set_yticklabels(lbls)
axes[2].set_xlabel('Predicted'); axes[2].set_ylabel('True')
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        axes[2].text(j,i,str(cm[i,j]),ha='center',va='center',
                     color='white' if cm[i,j]>cm.max()/2 else 'black')
plt.colorbar(im, ax=axes[2])

plt.tight_layout()
plt.savefig(str(FIGS/'evaluation_results.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Save & Download ──
results = pd.DataFrame([{'method':'CL-SDRG','sfr':sfr_val,**cls_m,**ret_m}]+abl)
results.to_csv('outputs/benchmark_results.csv', index=False)
pd.DataFrame([{'sfr':sfr_val,'target':SFR_TARGET,'passed':sfr_pass}]).to_csv('outputs/sfr_audit.csv', index=False)
print('📄 Results saved to outputs/')

try:
    from google.colab import files
    files.download('outputs/benchmark_results.csv')
    files.download(str(FIGS/'evaluation_results.png'))
    files.download(str(FIGS/'training_curves.png'))
except: pass

print(f'\n{"🎉"*10}')
print('  ALL 4 PHASES COMPLETE!')
print(f'{"🎉"*10}')